In [1]:
!pip install telethon
!pip install python-dotenv

In [1]:
import asyncio
import datetime as dt
import os
import json
import requests
#from dotenv import load_dotenv
from datetime import timedelta
from time import time
from telethon import TelegramClient

In [4]:
#get apip key
"""load_dotenv("key.env")  

JSEARCH_URL = "https://jsearch.p.rapidapi.com/search"
JSEARCH_HEADERS = {
    "X-RapidAPI-Key": os.getenv("JSEARCH_API_KEY"),
    "X-RapidAPI-Host": "jsearch.p.rapidapi.com"
}

TELEGRAM_API_ID   = int(os.getenv("TELEGRAM_API_ID"))
TELEGRAM_API_HASH = os.getenv("TELEGRAM_API_HASH")
TELEGRAM_CHANNELS = ["t.me/JobHitchpt", "t.me/SearchForJob", "t.me/nextjobs"]"""

JOB_PORTAL_API = "https://singaporejobs.com.sg/api/v1/front/assignments/open?key_words=&page_size=50"

In [4]:
# fetch from rapidapi
def fetch_jsearch_jobs(
    query: str = "software engineer in Singapore",
    page: int = 5,
    num_pages: int = 5,
    language: str = "en",
    output_path: str = "jsearch_output.json",
) -> list[dict]:
  
    params = {
        "query": query,
        "page": page,
        "num_pages": num_pages,
        "language": language,
    }

    response = requests.get(JSEARCH_URL, headers=JSEARCH_HEADERS, params=params)
    response.raise_for_status()

    data = response.json().get("data", [])
    print(f"[JSearch] Returned {len(data)} jobs.")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4, default=str)
    print(f"[JSearch] Saved to {output_path}")

    return data


In [5]:
async def fetch_telegram_jobs(
    channels: list[str] = TELEGRAM_CHANNELS,
    hours_back: int = 24,
    session_name: str = "session",
    output_path: str = "telescraping_output.json",
) -> list[dict]:
  
    messages = []
    since = dt.datetime.now() - timedelta(hours=hours_back)

    client = TelegramClient(session_name, TELEGRAM_API_ID, TELEGRAM_API_HASH)
    await client.start()

    for channel in channels:
        count = 0
        async for message in client.iter_messages(channel, offset_date=since, reverse=True):
            msg = message.to_dict()
            msg["channel_found"] = channel
            messages.append(msg)
            count += 1
        print(f"[Telegram] {channel}: {count} messages fetched.")

    await client.disconnect()

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(messages, f, ensure_ascii=False, indent=4, default=str)
    print(f"[Telegram] Saved {len(messages)} messages to {output_path}")

    return messages

In [9]:
# fetch from one known job board with exposed API

def get_from_job_portal():
    api_result = requests.get(JOB_PORTAL_API)
    if api_result.status_code != 200:
        print("Failed!")
        return []
    data = api_result.json()["results"]
    for entry in data:
        entry["description"] = "\n".join(entry["description"])
    with open(f"job-board-{time()}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4, default=str)
    return data

In [ ]:
jsearch_data  = fetch_jsearch_jobs()
telegram_data = await fetch_telegram_jobs()




[JSearch] Returned 47 jobs.
[JSearch] Saved to jsearch_output.json
[Telegram] t.me/JobHitchpt: 44 messages fetched.
[Telegram] t.me/SearchForJob: 10 messages fetched.
[Telegram] t.me/nextjobs: 21 messages fetched.
[Telegram] Saved 75 messages to telescraping_output.json


In [10]:
jb_data = get_from_job_portal()